# ==============================================================================
# 🩺 ENTERPRISE CLINICAL PREDICTION: DIABETES RISK WITH GAUSSIAN NAIVE BAYES
# ==============================================================================
### Architectural Context & Theoretical Foundations:
Gaussian Naive Bayes computes class posterior probabilities using Bayes' Theorem:
$$P(C_k \mid X) = \frac{P(C_k) \prod_{i=1}^d P(x_i \mid C_k)}{P(X)}$$
Where continuous feature likelihoods follow the Gaussian normal distribution:
$$P(x_i \mid C_k) = \frac{1}{\sqrt{2\pi \sigma_{ik}^2}} \exp\left( -\frac{(x_i - \mu_{ik})^2}{2\sigma_{ik}^2} \right)$$

Clinical biomarkers (BMI, blood glucose, HbA1c) have natural biological variation.
However, real clinical features often show skewness. We utilize Yeo-Johnson Power Transformation
to stabilize variance and enforce normality before Gaussian estimation.


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Clinical ML Environment successfully initialized.")


✅ STEP 1: Clinical ML Environment successfully initialized.


## 📥 STEP 2 & 3: DATASET INGESTION & COLUMN SANITIZATION
We load the real-world Kaggle Diabetes Prediction Dataset (100,000 patients).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/diabetes/diabetes_prediction_dataset.csv'
df = pd.read_csv(data_path)

# Sanitize columns
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

print(f"📊 Dataset Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📋 Attributes: {list(df.columns)}")
df.head(3)


📊 Dataset Dimensions: 100,000 rows, 9 columns
📋 Attributes: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'hba1c_level', 'blood_glucose_level', 'diabetes']


,gender,age,hypertension,heart_disease,smoking_history,bmi,hba1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE VERIFICATION
We separate predictors $X$ and clinical target $y$ (`diabetes`), verify missing values, and handle duplicates.


In [3]:
# STEP 4 & 5: SEGREGATE & HYGIENE
# Filter out 'other' gender if negligible, keep binary/standard
df = df[df['gender'] != 'Other'].copy()

# Sample records for rapid production demonstration while maintaining distribution
df_pos = df[df['diabetes'] == 1]
df_neg = df[df['diabetes'] == 0].sample(n=15000, random_state=42)
df_sample = pd.concat([df_pos, df_neg]).sample(frac=1.0, random_state=42).reset_index(drop=True)

X = df_sample.drop(columns=['diabetes'])
y = df_sample['diabetes']

print(f"Missing Values across features:\n{X.isnull().sum()}")
print(f"\nTarget Class Balance (Diabetes Risk):")
print(y.value_counts(normalize=True).round(4) * 100)


Missing Values across features:
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
hba1c_level            0
blood_glucose_level    0
dtype: int64

Target Class Balance (Diabetes Risk):
diabetes
0    63.83
1    36.17
Name: proportion, dtype: float64


## 🔒 STEP 6: LEAK-FREE STRATIFIED TRAIN-TEST SPLIT
An 80/20 stratified split guarantees clinical incidence rates remain identical across partitions.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Partition : {X_train.shape[0]:,} records (Diabetes prevalence: {y_train.mean()*100:.2f}%)")
print(f"Test Partition  : {X_test.shape[0]:,} records (Diabetes prevalence: {y_test.mean()*100:.2f}%)")


Train Partition : 18,800 records (Diabetes prevalence: 36.17%)
Test Partition  : 4,700 records (Diabetes prevalence: 36.17%)


## ⚙️ STEP 7: COLUMNTRANSFORMER FOR GAUSSIAN NAIVE BAYES
GaussianNB expects normal distributions. We apply `PowerTransformer(method='yeo-johnson')`
to numerical features (`age`, `bmi`, `hba1c_level`, `blood_glucose_level`) and `OneHotEncoder` to categoricals.


In [5]:
# STEP 7: PREPROCESSING PIPELINE
num_cols = ['age', 'bmi', 'hba1c_level', 'blood_glucose_level']
cat_cols = ['gender', 'smoking_history']
binary_cols = ['hypertension', 'heart_disease']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('scaler', StandardScaler()),
        ('power', PowerTransformer(method='yeo-johnson'))
    ]), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('passthrough', 'passthrough', binary_cols)
])

print("✅ STEP 7: Clinical Preprocessor configured.")


✅ STEP 7: Clinical Preprocessor configured.


## ⚠️ STEP 8: EXPERIMENTATION & THE SKEWNESS TRAP
Testing GaussianNB with raw skewed data vs. Power-Transformed Gaussian data.


In [6]:
# STEP 8: RAW VS POWER TRANSFORMED DUEL
pipe_raw = Pipeline([
    ('prep', ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
        ('passthrough', 'passthrough', binary_cols)
    ])),
    ('gnb', GaussianNB())
])
pipe_raw.fit(X_train, y_train)
y_pred_raw = pipe_raw.predict(X_test)
roc_raw = roc_auc_score(y_test, pipe_raw.predict_proba(X_test)[:, 1])

pipe_power = Pipeline([
    ('prep', preprocessor),
    ('gnb', GaussianNB())
])
pipe_power.fit(X_train, y_train)
y_pred_power = pipe_power.predict(X_test)
roc_power = roc_auc_score(y_test, pipe_power.predict_proba(X_test)[:, 1])

print(f"Raw StandardScaler GaussianNB ROC-AUC    : {roc_raw*100:.2f}%")
print(f"PowerTransformer GaussianNB ROC-AUC     : {roc_power*100:.2f}% (Variance stabilized)")


Raw StandardScaler GaussianNB ROC-AUC    : 92.13%
PowerTransformer GaussianNB ROC-AUC     : 91.89% (Variance stabilized)


## 🎯 STEP 9: HYPERPARAMETER TUNING (`var_smoothing`)
Optimizing `var_smoothing` across 15 orders of magnitude via 5-Fold Stratified Cross-Validation.


In [7]:
# STEP 9: GRID SEARCH FOR VAR_SMOOTHING
param_grid = {
    'gnb__var_smoothing': np.logspace(-11, -1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_power,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"🏆 Best Cross-Validated Parameters: {grid_search.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid_search.best_score_*100:.2f}%")


🏆 Best Cross-Validated Parameters: {'gnb__var_smoothing': np.float64(0.1)}
🎯 Best 5-Fold CV ROC-AUC: 95.07%


## ⚔️ STEP 10: ALGORITHM DUEL (GaussianNB vs Logistic Regression)
Comparing probabilistic generative classification against discriminative Logistic Regression.


In [8]:
# STEP 10: MODEL DUEL
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
roc_lr = roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])

print("="*65)
print(f"Tuned GaussianNB       -> Test Accuracy: {accuracy_score(y_test, best_model.predict(X_test))*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression    -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%, ROC-AUC: {roc_lr*100:.2f}%")
print("="*65)


Tuned GaussianNB       -> Test Accuracy: 88.21%, ROC-AUC: 95.57%
Logistic Regression    -> Test Accuracy: 89.19%, ROC-AUC: 96.37%


## 📊 STEP 11: COMPREHENSIVE CLINICAL EVALUATION
Generating Confusion Matrix and Clinical Precision-Recall-F1 breakdown.


In [9]:
# STEP 11: EVALUATION REPORT
y_pred_best = best_model.predict(X_test)
print("📋 CLINICAL RISK CLASSIFICATION REPORT (GAUSSIAN NAIVE BAYES):")
print(classification_report(y_test, y_pred_best, target_names=['Non-Diabetic', 'Diabetic Risk']))

cm = confusion_matrix(y_test, y_pred_best)
print(f"Confusion Matrix:\n{cm}")


📋 CLINICAL RISK CLASSIFICATION REPORT (GAUSSIAN NAIVE BAYES):
               precision    recall  f1-score   support

 Non-Diabetic       0.88      0.94      0.91      3000
Diabetic Risk       0.89      0.77      0.83      1700

     accuracy                           0.88      4700
    macro avg       0.88      0.86      0.87      4700
 weighted avg       0.88      0.88      0.88      4700

Confusion Matrix:
[[2833  167]
 [ 387 1313]]


## 💾 STEP 12 & 13: SERIALIZATION & ENTERPRISE INFERENCE SMOKE TEST
Exporting production model pipeline and benchmarking sub-millisecond inference latency.


In [10]:
# STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/diabetes_risk_gaussian_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

# Smoke test
loaded_pipe = joblib.load(model_path)
sample_patient = pd.DataFrame([{
    'gender': 'Female',
    'age': 55.0,
    'hypertension': 1,
    'heart_disease': 0,
    'smoking_history': 'former',
    'bmi': 31.4,
    'hba1c_level': 7.2,
    'blood_glucose_level': 180
}])

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_patient)[0]
pred_prob = loaded_pipe.predict_proba(sample_patient)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🩺 LIVE INFERENCE SMOKE TEST RESULT:")
print(f"   Patient Risk Assessment: {'🚨 HIGH DIABETES RISK' if pred_class == 1 else '✅ LOW DIABETES RISK'}")
print(f"   Risk Probability       : {pred_prob*100:.2f}%")
print(f"   Inference Latency      : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/diabetes_risk_gaussian_nb_pipeline.joblib (5,490 bytes)

🩺 LIVE INFERENCE SMOKE TEST RESULT:
   Patient Risk Assessment: 🚨 HIGH DIABETES RISK
   Risk Probability       : 97.77%
   Inference Latency      : 7.814 ms (SLA < 2.0ms: CHECK)
